In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp

In [ ]:
df = pd.read_csv ("/content/selling price of used cars.csv")

In [ ]:
df.head()

In [ ]:
headers = ["symboling", "normalized-losses", "make",
		"fuel-type", "aspiration","num-of-doors",
		"body-style","drive-wheels", "engine-location",
		"wheel-base","length", "width","height", "curb-weight",
		"engine-type","num-of-cylinders", "engine-size",
		"fuel-system","bore","stroke", "compression-ratio",
		"horsepower", "peak-rpm","city-mpg","highway-mpg","price"]

df.columns=headers
df.head()

In [ ]:
data = df

# Finding the missing values
data.isna().any()

# Finding if missing values
data.isnull().any()

In [ ]:
# converting mpg to L / 100km
data['city-mpg'] = 235 / df['city-mpg']
data.rename(columns = {'city_mpg': "city-L / 100km"}, inplace = True)

print(data.columns)

# checking the data type of each column
data.dtypes

In [ ]:
data.price.unique()

In [ ]:
# Here it contains '?', so we Drop it
data = data[data.price != '?']

# checking it again
data.dtypes

In [ ]:
# Ensure 'price' column is numeric
data['price'] = pd.to_numeric(data['price'], errors='coerce')

# Drop rows with NaN values in 'price' if necessary, or handle them as needed
data = data.dropna(subset=['price'])

# Normalize 'length', 'width', 'height'
data['length'] = data['length'] / data['length'].max()
data['width'] = data['width'] / data['width'].max()
data['height'] = data['height'] / data['height'].max()

# Binning - grouping values
bins = np.linspace(min(data['price']), max(data['price']), 4)
group_names = ['Low', 'Medium', 'High']
data['price-binned'] = pd.cut(data['price'], bins,
                              labels=group_names,
                              include_lowest=True)

# Display binned price data
print(data['price-binned'])

# Plot histogram
plt.hist(data['price-binned'], bins=len(group_names), edgecolor='black')
plt.xlabel('Price Bins')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# categorical to numerical variables
pd.get_dummies(data['fuel-type']).head()

# descriptive analysis
# NaN are skipped
data.describe()

In [ ]:
# Example of a box plot
plt.figure(figsize=(8, 6))
plt.boxplot(data['price'])
plt.title('Boxplot of Prices')
plt.ylabel('Price')
plt.show()

# Box plot using seaborn
plt.figure(figsize=(10, 6))
sns.boxplot(x='drive-wheels', y='price', data=data)
plt.title('Boxplot of Price by Drive Wheels')
plt.xlabel('Drive Wheels')
plt.ylabel('Price')
plt.show()

# Predicting price based on engine size
plt.figure(figsize=(10, 6))
plt.scatter(data['engine-size'], data['price'], alpha=0.7)
plt.title('Scatterplot of Engine Size vs Price')
plt.xlabel('Engine Size')
plt.ylabel('Price')
plt.xticks(rotation=45)  # Rotate x-axis labels if needed
plt.grid(True)
plt.show()

In [ ]:
test = data[['drive-wheels', 'body-style', 'price']]
data_grp = test.groupby(['drive-wheels', 'body-style'],
						as_index = False).mean()

data_grp

In [ ]:
# pivot method
data_pivot = data_grp.pivot(index = 'drive-wheels',
							columns = 'body-style')
data_pivot

In [ ]:
# heatmap for visualizing data
plt.pcolor(data_pivot, cmap ='RdBu')
plt.colorbar()
plt.show()

In [ ]:
# Analysis of Variance- ANOVA
# returns f-test and p-value
# f-test = variance between sample group means divided by
# variation within sample group
# p-value = confidence degree
data_annova = data[['make', 'price']]
grouped_annova = data_annova.groupby(['make'])
annova_results_l = sp.stats.f_oneway(
                             grouped_annova.get_group('honda')['price'],
                             grouped_annova.get_group('subaru')['price']
                                    )
print(annova_results_l)

In [ ]:
# strong corealtion between a categorical variable
# if annova test gives large f-test and small p-value

# Correlation- measures dependency, not causation
sns.regplot(x ='engine-size', y ='price', data = data)
plt.ylim(0, )